# Final Project — AI Security Analyst for Your Own Lab

**Two-person capstone • Course: Applied AI in Cybersecurity**

> **Academic integrity:** Execute this notebook against *your own* isolated lab, save actual outputs, run both model backends, and use human review before citing CVEs. The blank cells are deliberate; no scan or model output has been fabricated. The reference notebook files are teaching examples, not experimental results.

### Research questions
1. What does Chain report from memory without advisory retrieval?
2. Does retrieval improve the grounded coverage of *observed* findings?
3. Can an MCP agent select tools safely with two independent mandatory brakes?
4. How do hosted and paid models compare in accuracy, time, cost, and data transmission?


## 0 — Environment and safety

Run this notebook with the working directory set to `final_project/`. All tool calls are scoped to exact configured IPs/ports. **Every actual scan requires approval**, and the agent requires a fresh approval for each action. The service lab runs on host-only VM; Juice Shop is bound to loopback.

In [ ]:
from pathlib import Path
import json, os, sys, subprocess, time, asyncio
ROOT = Path.cwd()
assert (ROOT / 'guardrails/targets.json').exists(), 'Launch Jupyter from final_project/'
from guardrails.scope import check_target, check_url, ScopeError
from guardrails.approval import require_approval
from knowledge_base.loader import read_sections
from evaluation.coverage import calculate_coverage
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT/'.env')
except ImportError:
    print('python-dotenv not installed; use environment variables instead')
print('KB sections loaded:',len(read_sections()))


## 1 — Fail-closed scope demonstration

The following blocked external IP must be refused without opening a socket. Verify real allowed local target separately after configuring `guardrails/targets.json`.

In [ ]:
try:
    check_target('8.8.8.8',53)
    raise AssertionError('BLOCKING FAILURE')
except ScopeError as exc:
    print('Expected out-of-scope rejection:',exc)
print('No network action was attempted.')


## 2 — Real web lab scan (Partner B)

Start `docker compose -f labs/web_lab/compose.yaml up -d` on this same machine. Enter `APPROVE` individually for each test. Missing headers are observations, not proof of a vulnerability.

In [ ]:
from scanners.web.security_headers import check_security_headers
from scanners.web.http_methods import check_http_methods
from scanners.web.robots_checker import check_robots
WEB_URL='http://127.0.0.1:3000'
check_url(WEB_URL)
web_findings=[]
for label,scanner in [('headers',check_security_headers),('methods',check_http_methods),('robots.txt',check_robots)]:
    if require_approval(f'Inspect {label} on {WEB_URL}'):
        web_findings.extend(scanner(WEB_URL))
print(json.dumps(web_findings,indent=2))
(ROOT/'results/web_scans/live_findings.json').write_text(json.dumps(web_findings,indent=2))


## 3 — Real service lab scan (Partner A)

Change `SERVICE_IP` to the actual isolated host-only VM IP and edit `guardrails/targets.json` accordingly. This step probes three real scanners without exploitation. The sample address is not a claim that such a VM exists.

In [ ]:
from scanners.service.port_scanner import scan_ports
from scanners.service.banner_grabber import grab_banner
from scanners.service.ftp_checker import check_anonymous_ftp
SERVICE_IP='192.168.56.101'  # REPLACE after verifying the actual host-only VM
PORTS=[21,22,80]
service_findings=[]
for port in PORTS:
    check_target(SERVICE_IP,port)
if require_approval(f'TCP connect scan {SERVICE_IP}:{PORTS}'):
    service_findings.extend(scan_ports(SERVICE_IP,PORTS))
if require_approval(f'Banner read {SERVICE_IP}:21'):
    service_findings.extend(grab_banner(SERVICE_IP,21))
if require_approval(f'Anonymous FTP login check {SERVICE_IP}:21'):
    service_findings.extend(check_anonymous_ftp(SERVICE_IP,21))
print(json.dumps(service_findings,indent=2))
(ROOT/'results/service_scans/live_findings.json').write_text(json.dumps(service_findings,indent=2))


## 4 — RAG coverage candidates

Retrieve advisory sections for each finding. **A candidate match is not yet a verified grounded citation.** Each student should review the source and mark whether the finding is actually supported. Expand both KBs and rerun to measure before/after.

In [ ]:
from agent.retrieval import grounding
combined_findings=service_findings+web_findings
candidates=grounding(combined_findings)
for hit in candidates:
    f=hit['finding']
    print(f['scanner'],f['category'],'=>',[(d['source'],d['title']) for d in hit['candidates']])
(ROOT/'results/reports/retrieval_candidates.json').write_text(json.dumps(candidates,indent=2))


## 5 — Chain versus RAG on the SAME evidence

This cell requires a real configured hosted LLM (and later the paid API model). Both use temperature 0. The Chain has no advisory text; RAG receives retrieved sections. The experiment records actual responses and API usage.

In [ ]:
from models.model_factory import get_model,ModelConfigurationError
from agent.chain import run_chain
from agent.retrieval import run_retrieval
# Change to 'API' after running HOSTED to execute the exact same experiment.
BACKEND='HOSTED'
if not combined_findings:
    print('Collect lab findings before running both stages.')
else:
    try:
        model=get_model(BACKEND)
        chain_result=run_chain(model,combined_findings)
        rag_result=run_retrieval(model,combined_findings)
        print('CHAIN REPORT:',chain_result['report'])
        print('\nGROUNDED REPORT:',rag_result['report'])
        for name,result in [('chain',chain_result),('rag',rag_result)]:
            (ROOT/f'results/reports/{name}_{BACKEND.lower()}.json').write_text(json.dumps(result,indent=2))
    except ModelConfigurationError as exc:
        print('Not configured:',exc)


## 6 — Actual MCP tool discovery and approved agent action

The agent proposes exactly one tool in JSON. Agent proposals are validated **before** asking for approval; the MCP server checks scope again. Run the next cell only after the MCP SDK is installed and the hosted model is configured.

In [ ]:
from agent.security_agent import run_agent_step
# Explicit interactive step. Commented by default to avoid accidental actions
# in 'Run All'; uncomment for the recorded live demonstration.
# outcome=await run_agent_step(get_model('HOSTED'),
#     'Inspect security headers for http://127.0.0.1:3000')
# print(json.dumps(outcome,indent=2))


## 7 — Hosted model versus paid API comparison

Compare with identical prompts and fixed observations. Record measured latency, token usage, actual invoiced cost and whether any lab data leaves your controlled environment. Do not invent cost, GPU specs or correctness percentages.

In [ ]:
from evaluation.compare_models import compare_models
from agent.prompts import CHAIN_SYSTEM
prompt='Analyze these recorded observations. Label unverified claims.\n'+json.dumps(combined_findings)
# Uncomment once BOTH providers are live:
# comparisons=compare_models({'hosted':get_model('HOSTED'),'paid_api':get_model('API')},CHAIN_SYSTEM,prompt)
# (ROOT/'results/metrics/model_comparison.json').write_text(json.dumps(comparisons,indent=2))
print('Manual accuracy review required; see evaluation/README.md')


## 8 — Human verification and reflection

Populate reviewer assessments from **actual** source checks. Example structure:
`{'finding_id': 0, 'verified_grounded': True, 'review_note':'Verified against ...'}`.
Do not count a retrieved document as grounded until checked.

In [ ]:
reviewed=[]  # Fill after inspecting observations and original advisories
print('Reviewed coverage:',calculate_coverage(reviewed))
print('Report outline: docs/report_template.md')
print('Video guide: docs/video_demo.md')
